# Tutorial 23 -- Analysis, Fitting, and Result Extraction

Use the `cqed_sim.calibration_targets` helpers to generate fitted calibration summaries and inspect the returned `CalibrationResult` objects.

**Prerequisites.** Tutorials 09, 11, and 12 are recommended first.

**Scope note.** The `calibration_targets` module is a lightweight convenience layer that returns fitted curves and metadata; it is not a full pulse-level experimental loop.


## 1. Goal

We will call the public calibration-target helpers, inspect the fitted parameters and uncertainties, and visualize the returned raw data.


## 2. Physical Background

A common workflow is to turn simulated or measured traces into a small set of fit parameters. `cqed_sim` exposes a convenience API for that pattern through `CalibrationResult` objects.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    cross_kerr_conditional_phase,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    gaussian_quasistatic_echo_excited_population,
    gaussian_quasistatic_ramsey_excited_population,
    ns,
    ramsey_population,
    resonant_drive_excited_population,
    t1_relaxation_population,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=MHz(-220.0),
    chi=MHz(-2.2),
    kerr=0.0,
    n_cav=8,
    n_tr=3,
)

spectroscopy_frequencies = np.linspace(model.omega_q - MHz(12.0), model.omega_q + MHz(12.0), 201)
rabi_amplitudes = np.linspace(0.0, 1.4, 81)
delays = np.linspace(0.0, 25.0 * us, 101)


## 5. Model Construction


In [ ]:
spectroscopy = run_spectroscopy(model, spectroscopy_frequencies)
rabi = run_rabi(model, rabi_amplitudes, duration=40.0 * ns, omega_scale=2.0 * np.pi * 12.0e6)
t1 = run_t1(model, delays, t1=18.0 * us)
ramsey = run_ramsey(model, delays, detuning=2.0 * np.pi * 0.6e6, t2_star=8.0 * us)
echo = run_t2_echo(model, delays, t2_echo=14.0 * us)


## 6. Pulse / Sequence Construction


In [ ]:
pi_amplitude = np.pi / (rabi.fitted_parameters["omega_scale"] * rabi.fitted_parameters["duration"])
summary = {
    "omega_01_mhz": angular_to_mhz(spectroscopy.fitted_parameters["omega_01"]),
    "omega_12_mhz": angular_to_mhz(spectroscopy.fitted_parameters["omega_12"]),
    "omega_scale": rabi.fitted_parameters["omega_scale"],
    "pi_amplitude": pi_amplitude,
    "t1_us": t1.fitted_parameters["t1"] / us,
    "delta_omega_mhz": angular_to_mhz(ramsey.fitted_parameters["delta_omega"]),
    "t2_star_us": ramsey.fitted_parameters["t2_star"] / us,
    "t2_echo_us": echo.fitted_parameters["t2_echo"] / us,
}
summary


## 7. Running the Simulation


In [ ]:
print("Spectroscopy fit:", spectroscopy.fitted_parameters)
print("Rabi fit:", rabi.fitted_parameters)
print("T1 fit:", t1.fitted_parameters)
print("Ramsey fit:", ramsey.fitted_parameters)
print("Echo fit:", echo.fitted_parameters)


## 8. Visualizing the Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
axes[0].plot(
    (spectroscopy.raw_data["drive_frequencies"] - model.omega_q) / (2.0 * np.pi * 1.0e6),
    spectroscopy.raw_data["response"],
)
axes[0].set_xlabel("Drive frequency relative to bare omega_q [MHz]")
axes[0].set_ylabel("Spectroscopy response")
axes[0].set_title("Calibration-target spectroscopy trace")

axes[1].plot(rabi.raw_data["amplitudes"], rabi.raw_data["excited_population"])
axes[1].set_xlabel("Normalized drive amplitude")
axes[1].set_ylabel(r"Excited population")
axes[1].set_title("Calibration-target Rabi trace")
plt.show()


## 9. Physical Interpretation

The convenience fit objects are useful when you want a fast notebook summary or a compact developer-facing API. They are especially handy in workflow notebooks where the goal is to collect a few calibration targets instead of replaying every low-level pulse step. The spectroscopy helper reports absolute angular transition frequencies, while the Ramsey helper reports the fitted angular detuning under the explicit key `delta_omega`; the plot above shows the spectroscopy sweep re-expressed relative to the bare `omega_q` for readability.


## 10. Exercises / Next Steps

- Compare these fitted outputs to the explicit pulse-level notebooks from Tutorials 09 through 13.
- Use the returned `summary` dictionary as input to your own higher-level workflow code.
- Continue to Tutorial 25 for a compact end-to-end calibration example.
